In [1]:
# 09_hourly_predictions.py — hourly zonal predictions for the dashboard.
# Same trusted eval path as 08; banks the FULL hourly forecast, not daily sums.
# Output: reports/tft_hourly_predictions.parquet
#   columns: issue_date, lead_day, ts_utc, zone, pred_mw, actual_mw
import importlib.util, os
import numpy as np, pandas as pd, torch

ROOT = "/opt/app-root/src/Forecasting-Energy-Demand/Sangar"
EVAL = os.path.join(ROOT, "06_eval_refit2023_fx.py")
OUT  = os.path.join(ROOT, "..", "reports", "tft_hourly_predictions.parquet")

spec = importlib.util.spec_from_file_location("evalmod", EVAL)
ev = importlib.util.module_from_spec(spec); spec.loader.exec_module(ev)

def cache(d): return os.path.join(ROOT, f"hourly_lead{d}.npz")

def main():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    df = ev.load_base()
    leads = pd.read_parquet(ev.LEADS); leads.index = pd.to_datetime(leads.index)
    if getattr(leads.index, "tz", None) is not None:
        leads.index = leads.index.tz_localize(None)

    one = df[df["zone"] == df["zone"].iloc[0]][["time_idx", "utc"]]
    idx2utc = dict(zip(one["time_idx"], one["utc"]))

    tsi = int(df.loc[df["utc"] >= ev.TEST_START, "time_idx"].min())
    training_ds = ev.build_training_ds(df)
    model = ev.TemporalFusionTransformer.load_from_checkpoint(ev.CKPT).to(device).eval()

    df = df[df["time_idx"] >= tsi - ev.ENCODER_LEN - 1].copy()
    ZONES = sorted(df["zone"].unique())
    print(f"windows from idx {tsi}, device={device}", flush=True)

    frames = []
    for d in range(1, 6):
        print(f"\n=== Lead {d} ===", flush=True)
        s, e = (d - 1) * 24, d * 24
        if os.path.exists(cache(d)):
            z = np.load(cache(d), allow_pickle=True)
            starts, P, A = z["starts"], z["pred"], z["act"]
            print("  cached", flush=True)
        else:
            dfx = ev.swap_lead(df, leads, d)
            hat, true = {}, {}
            for zi, zone in enumerate(ZONES):
                yh, yt, tt = ev.predict_one_zone(model, training_ds,
                                                 dfx[dfx["zone"] == zone], tsi, device)
                for i, t in enumerate(tt):
                    hat.setdefault(int(t), {})[zone] = yh[i][s:e]
                    true.setdefault(int(t), {})[zone] = yt[i][s:e]
                print(f"    {zone:10s} ({zi+1}/11)", flush=True)
            del dfx
            good = sorted(t for t, v in hat.items() if len(v) == 11)
            starts = np.array(good)
            P = np.stack([np.stack([hat[t][z] for z in ZONES]) for t in good])  # (n,11,24)
            A = np.stack([np.stack([true[t][z] for z in ZONES]) for t in good])
            np.savez_compressed(cache(d), starts=starts, pred=P, act=A)
            del hat, true

        # only windows whose decoder starts at local midnight -> one issue date each
        for i, t in enumerate(starts):
            utc0 = idx2utc[int(t)]
            local = pd.Timestamp(utc0, tz="UTC").tz_convert("America/New_York")
            if local.hour != 0:
                continue
            hours = pd.date_range(pd.Timestamp(utc0) + pd.Timedelta(hours=s),
                                  periods=24, freq="h")
            for zi, zone in enumerate(ZONES):
                frames.append(pd.DataFrame({
                    "issue_date": local.date(),
                    "lead_day": d,
                    "ts_utc": hours,
                    "zone": zone,
                    "pred_mw": P[i, zi].astype("float32"),
                    "actual_mw": A[i, zi].astype("float32"),
                }))

    out = pd.concat(frames, ignore_index=True)
    os.makedirs(os.path.dirname(OUT), exist_ok=True)
    out.to_parquet(OUT, index=False)
    print(f"\nwrote {len(out):,} rows, {out['issue_date'].nunique()} issue dates -> {os.path.abspath(OUT)}", flush=True)

    # sanity: hourly MAPE per lead should match the deck (day1 ~2.2%, overall ~2.95%)
    tot = (out.groupby(["lead_day", "ts_utc"])[["pred_mw", "actual_mw"]].sum().reset_index())
    for d in range(1, 6):
        g = tot[tot["lead_day"] == d]
        m = (np.abs(g["actual_mw"] - g["pred_mw"]) / g["actual_mw"]).mean() * 100
        print(f"  day{d} hourly statewide MAPE: {m:.2f}%", flush=True)

if __name__ == "__main__":
    main()

/opt/app-root/lib64/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/opt/app-root/lib64/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.


windows from idx 75080, device=cuda

=== Lead 1 ===
  fallback cells: 0
    CAPITL     (1/11)
    CENTRL     (2/11)
    DUNWOD     (3/11)
    GENESE     (4/11)
    HUD VL     (5/11)
    LONGIL     (6/11)
    MHK VL     (7/11)
    MILLWD     (8/11)
    N.Y.C.     (9/11)
    NORTH      (10/11)
    WEST       (11/11)

=== Lead 2 ===
  fallback cells: 0
    CAPITL     (1/11)
    CENTRL     (2/11)
    DUNWOD     (3/11)
    GENESE     (4/11)
    HUD VL     (5/11)
    LONGIL     (6/11)
    MHK VL     (7/11)
    MILLWD     (8/11)
    N.Y.C.     (9/11)
    NORTH      (10/11)
    WEST       (11/11)

=== Lead 3 ===
  fallback cells: 0
    CAPITL     (1/11)
    CENTRL     (2/11)
    DUNWOD     (3/11)
    GENESE     (4/11)
    HUD VL     (5/11)
    LONGIL     (6/11)
    MHK VL     (7/11)
    MILLWD     (8/11)
    N.Y.C.     (9/11)
    NORTH      (10/11)
    WEST       (11/11)

=== Lead 4 ===
  fallback cells: 0
    CAPITL     (1/11)
    CENTRL     (2/11)
    DUNWOD     (3/11)
    GENESE     (4/11)
